# 10 — ImageNet-C: cache logit (GPU)

**Kenapa notebook ini ada.** `topvenue_gap_audit.md` A3: ImageNet-C adalah jembatan
ke paper UM-TTA. Dua pertanyaan berbeda dijawab di sini, dan keduanya butuh dump
logit yang sama:

1. **PCC di bawah pergeseran distribusi** — kalibrasi pada citra bersih, evaluasi
   pada citra terkorupsi. Ini yang membuat F4 (exchangeability) wajib ditulis.
2. **Hasil negatif UM-TTA** — TTA tidak memperbaiki cakupan per-kelas. Backbone-nya
   sama persis (torchvision ResNet-50), jadi aturan keras `AGENTS.md` terpenuhi.

**Satu hal yang harus diketahui sebelum menjalankan.** `unified_eval_corruption.py`
di repo UM-TTA **tidak pernah menyimpan logit ke disk** — ia mengekstrak, menilai di
memori, lalu hanya menyimpan `theta_global.pt`. Jadi menjalankan skrip itu apa adanya
akan menjawab pertanyaan UM-TTA dan meninggalkan PCC tanpa apa pun untuk dibaca.
Notebook ini yang melakukan penyimpanan, tetapi **setiap langkah sisi-model**
(augmentasi, agregasi TTA, suhu Platt) tetap memanggil fungsi repo itu, bukan tulisan
ulang. Tata letak berkasnya identik dengan notebook 07, jadi loader notebook 08 dan
notebook 12 membacanya tanpa perubahan.

| | |
|---|---|
| Unduhan | ~6,7 GB (val bersih) + ~24 GB (4 tar ImageNet-C) |
| Disk puncak | ~15 GB — tar dihapus segera setelah diekstrak |
| GPU | ~45 menit di L4, ~75 menit di T4 |
| Bisa dilanjutkan | ya — tiap (korupsi, severity, metode, seed) dilewati bila cache-nya sudah ada di Drive |

## 1. Config

In [ ]:
# === EDIT ME ===========================================================
DRIVE_ROOT = '/content/drive/MyDrive/pcc'
UMTTA_URL  = 'https://github.com/octadion/umtta_conformal.git'
UMTTA_DIR  = '/content/umtta'

VAL_TAR = 'https://image-net.org/data/ILSVRC/2012/ILSVRC2012_img_val.tar'
LBL_URL = ('https://github.com/tensorflow/models/raw/master/research/slim/'
           'datasets/imagenet_2012_validation_synset_labels.txt')
VAL_DIR = '/content/imagenet_val'

# Satu korupsi per KATEGORI Zenodo, supaya empat jenis pergeseran terwakili tanpa
# mengunduh lima tar. Nama tar-nya yang menentukan biaya unduhan, bukan nama
# korupsinya, jadi memilih dua korupsi dari kategori yang sama akan gratis --
# tetapi tidak menambah keragaman pergeseran, yang justru intinya.
ZENODO = 'https://zenodo.org/records/2235448/files/{}.tar?download=1'
CORRUPTIONS = [('gaussian_noise',    'noise'),
               ('defocus_blur',      'blur'),
               ('snow',              'weather'),
               ('jpeg_compression',  'digital')]
SEVERITIES = (3, 5)          # baseline: dua tingkat, supaya ada kemiringan

# Protokol split sama dengan unified_eval_corruption.py: 25.000 bersih dibagi
# 20% D_TTA / 80% D_cal, sisanya dibuang; test = citra terkorupsi.
NUM_CAL   = 25000
SEEDS     = (0, 1, 2)
BATCH_SIZE = 128
NUM_AUG    = 3               # NUM_AUGMENTATIONS di repo
TTA_EPOCHS, TTA_LR = 10, 0.01

# baseline itu deterministik: logitnya tidak bergantung seed, jadi dihitung SEKALI
# untuk seluruh 50.000 dan tiap seed hanya mengindeksnya. Itu yang membuat lengan
# ini murah, dan itu juga lengan utama PCC (50 baris eval/kelas -> regime A).
DO_UMTTA         = True
UMTTA_SEVERITIES = (5,)      # pergeseran terkuat: di situ hasil negatifnya paling jelas
UMTTA_TEST_N     = 10000     # 10 citra/kelas, terstratifikasi -- lihat CAVEAT di sel akhir

SAVE_ROOT = '/content/results/imagenetc_resnet50'
DEST_ROOT = DRIVE_ROOT + '/umtta/imagenetc_resnet50'
# =======================================================================
print('korupsi', [c for c, _ in CORRUPTIONS], '| severity', SEVERITIES)
print('umtta', DO_UMTTA, '| severity umtta', UMTTA_SEVERITIES, '| n test', UMTTA_TEST_N)

## 2. Mount, repo, GPU

In [ ]:
import os, subprocess, sys, glob, time, json, shutil
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(DEST_ROOT, exist_ok=True)

if not os.path.isdir(UMTTA_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', UMTTA_URL, UMTTA_DIR], check=True)
print(subprocess.run(['git', 'log', '--oneline', '-1'], cwd=UMTTA_DIR,
                     capture_output=True, text=True).stdout.strip())
subprocess.run(['pip', 'install', '-q', '-r',
                os.path.join(UMTTA_DIR, 'requirements.txt')], check=False)
if UMTTA_DIR not in sys.path:
    sys.path.insert(0, UMTTA_DIR)

import torch, numpy as np
import torch.utils.data as tdata
import torchvision
import torchvision.transforms as transforms
assert torch.cuda.is_available(), 'notebook ini butuh GPU'
print('torch', torch.__version__, '|', torch.cuda.get_device_name(0))

# fungsi-fungsi METODE diambil dari repo, tidak ditulis ulang
from umtta_conformal import (get_tta_logits_dataset, learn_global_tta_weights,
                             learn_umtta_parameters)
from conformal_engine import extract_temperature
print('fungsi repo terimpor: get_tta_logits_dataset, learn_global_tta_weights,',
      'learn_umtta_parameters, extract_temperature')

## 3. Val bersih — unduh dan tata ke folder WNID

Sama persis dengan notebook 07: `ILSVRC2012_img_val.tar` isinya 50.000 JPEG datar,
sedangkan `ImageFolder` butuh subfolder per kelas. Hasil penataan **di-assert**,
bukan dicetak — kalau salah, semua di atasnya tidak sah.

In [ ]:
os.makedirs('/content/dl', exist_ok=True)
TAR = '/content/dl/ILSVRC2012_img_val.tar'
LBL = '/content/dl/val_synset_labels.txt'

if not os.path.exists(LBL) or os.path.getsize(LBL) < 100000:
    subprocess.run(['wget', '-q', LBL_URL, '-O', LBL], check=True)
lbls = [l.strip() for l in open(LBL) if l.strip()]
assert len(lbls) == 50000, 'label harus 50.000 baris, dapat ' + str(len(lbls))

if len(glob.glob(VAL_DIR + '/*/*.JPEG')) < 50000:
    if not os.path.exists(TAR) or os.path.getsize(TAR) < 6_000_000_000:
        subprocess.run(['apt-get', 'install', '-qq', 'aria2'], check=False)
        t0 = time.time()
        subprocess.run(['aria2c', '-x', '16', '-s', '16', VAL_TAR,
                        '-d', '/content/dl', '-o', 'ILSVRC2012_img_val.tar'],
                       check=True)
        print('unduh val {:.0f}s | {:.2f} GB'.format(
            time.time() - t0, os.path.getsize(TAR) / 1e9))
    RAW = '/content/dl/raw_val'
    os.makedirs(RAW, exist_ok=True)
    if len(glob.glob(RAW + '/*.JPEG')) < 50000:
        subprocess.run(['tar', '-xf', TAR, '-C', RAW], check=True)
    for w in set(lbls):
        os.makedirs(os.path.join(VAL_DIR, w), exist_ok=True)
    for i, w in enumerate(lbls, start=1):
        src = os.path.join(RAW, 'ILSVRC2012_val_%08d.JPEG' % i)
        if os.path.exists(src):
            os.rename(src, os.path.join(VAL_DIR, w, os.path.basename(src)))
    for p in (TAR, RAW):
        subprocess.run(['rm', '-rf', p], check=False)

dirs = sorted(d for d in os.listdir(VAL_DIR)
              if os.path.isdir(os.path.join(VAL_DIR, d)))
counts = [len(os.listdir(os.path.join(VAL_DIR, d))) for d in dirs]
assert len(dirs) == 1000, 'harus 1000 kelas, dapat ' + str(len(dirs))
assert sum(counts) == 50000, 'harus 50.000 gambar, dapat ' + str(sum(counts))
assert min(counts) == max(counts) == 50, 'tiap kelas harus 50 gambar'
print('val bersih OK: 1000 kelas x 50 =', sum(counts))

## 4. Model dan dataset bersih

Transform-nya disalin dari `unified_eval.py` baris 1253–1258 dan modelnya dari baris
1265, supaya logit di sini sebanding dengan logit notebook 07 — bukan mirip,
melainkan dari pipeline yang sama.

In [ ]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])])

model = torchvision.models.resnet50(pretrained=True).cuda().eval()
clean_ds = torchvision.datasets.ImageFolder(VAL_DIR, transform)
print('val bersih:', len(clean_ds), 'citra |', len(clean_ds.classes), 'kelas')
assert len(clean_ds) == 50000
CLEAN_CLASSES = list(clean_ds.classes)

## 5. Unduh + ekstrak satu korupsi — anggota tar DITEMUKAN, bukan ditebak

Tata letak di dalam tar ImageNet-C tidak seragam antar rilis: sebagian punya awalan
direktori, sebagian tidak. Menebak lalu mengekstrak 7 GB dengan pola yang tidak cocok
menghasilkan direktori kosong dan exit 0 — persis kegagalan senyap yang sudah pernah
kita alami dengan unduhan. Jadi beberapa anggota pertama dibaca dulu (murah: header
tar ada di awal berkas), polanya diturunkan dari situ, dan hasil ekstraksi dihitung.

Hanya severity yang dibutuhkan yang diekstrak, dan **tar dihapus begitu selesai** —
satu tar 7 GB di disk sekaligus, tidak pernah empat.

In [ ]:
DL = '/content/dl'
IC_ROOT = '/content/imagenet_c'
os.makedirs(IC_ROOT, exist_ok=True)

def sev_dir(corr, sev):
    return '{}/{}/{}'.format(IC_ROOT, corr, sev)

def n_images(d):
    # himpunan, bukan penjumlahan dua glob: pada berkas-sistem yang tidak peka
    # huruf besar-kecil kedua pola cocok ke berkas yang SAMA dan hitungannya jadi
    # dua kali lipat -- yang berarti pemeriksaan kelengkapan lolos padahal separuh.
    return len(set(glob.glob(d + '/*/*.JPEG')) | set(glob.glob(d + '/*/*.jpeg')))

def fetch(corr, category, sevs):
    """Pastikan {IC_ROOT}/{corr}/{sev} terisi untuk tiap sev. Kembalikan True bila siap."""
    need = [s for s in sevs if n_images(sev_dir(corr, s)) < 40000]
    if not need:
        print('  {} severity {} sudah ada'.format(corr, list(sevs)))
        return True
    tar = '{}/{}.tar'.format(DL, category)
    if not os.path.exists(tar) or os.path.getsize(tar) < 1_000_000_000:
        t0 = time.time()
        subprocess.run(['aria2c', '-x', '16', '-s', '16', '--allow-overwrite=true',
                        ZENODO.format(category), '-d', DL,
                        '-o', category + '.tar'], check=True)
        print('  unduh {}.tar {:.0f}s | {:.2f} GB'.format(
            category, time.time() - t0, os.path.getsize(tar) / 1e9))
    # --- ekstraksi lewat modul tarfile, bukan lewat perintah tar.
    # `tar --wildcards` hanya ada di GNU tar; bsdtar menolaknya, dan pola yang
    # tidak cocok menghasilkan direktori kosong dengan exit 0 -- kegagalan senyap
    # yang persis ingin dihindari. tarfile juga membuat kita bisa MENGHITUNG
    # anggota yang cocok, jadi 'nol yang cocok' langsung ketahuan di sini.
    import tarfile
    want = {str(s) for s in need}
    n_ext = 0
    t0 = time.time()
    with tarfile.open(tar) as tf:
        for m in tf:
            if not m.isfile():
                continue
            parts = m.name.split('/')
            if corr not in parts:
                continue
            i = parts.index(corr)
            if i + 2 >= len(parts) or parts[i + 1] not in want:
                continue
            # tata ulang ke {IC_ROOT}/{corr}/{sev}/{wnid}/berkas, apa pun awalan
            # di dalam tar-nya
            dst = os.path.join(IC_ROOT, *parts[i:])
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            src_f = tf.extractfile(m)
            if src_f is None:
                continue
            with open(dst, 'wb') as out:
                shutil.copyfileobj(src_f, out)
            n_ext += 1
    print('  ekstrak {} berkas dalam {:.0f}s'.format(n_ext, time.time() - t0))
    assert n_ext > 0, ('tidak ada anggota tar yang cocok untuk {} severity {} -- '
                       'tata letak tar berbeda dari yang diduga'.format(corr, need))
    ok = True
    for s in sevs:
        n = n_images(sev_dir(corr, s))
        print('  {} severity {}: {} citra'.format(corr, s, n))
        ok = ok and n >= 40000
    for p in glob.glob(DL + '/*.tar'):
        os.remove(p)
    return ok

print('fungsi unduh siap')

## 6. Pembungkus ekstraksi + penyimpan Drive

`get_tta_logits_dataset` repo yang menghitung logit; sel ini hanya menaruhnya di
berkas dan **melewati pekerjaan yang cache-nya sudah ada di Drive**. Itu yang membuat
notebook ini bisa dilanjutkan: runtime yang mati di korupsi ketiga tidak mengulang dua
yang pertama.

In [ ]:
def extract_logits(ds, method, params):
    """Logit lewat fungsi repo. tta_method dipetakan persis seperti unified_eval.py."""
    tta = {'baseline': 'none', 'tta_avg': 'avg',
           'tta_learned': 'learned', 'umtta': 'umtta'}[method]
    out = get_tta_logits_dataset(model, ds, NUM_AUG, tta_method=tta,
                                 tta_params=params, batch_size=BATCH_SIZE)
    return out.tensors[0], out.tensors[1].long()

def load_pt(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:            # torch < 1.13 tidak punya weights_only
        return torch.load(path, map_location='cpu')

def save_pt(path, logits, labels):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({'logits': logits, 'labels': labels}, path)
    return os.path.getsize(path)

def push(subdir, tag):
    """Salin satu subdirektori hasil ke Drive, verifikasi per berkas."""
    src, dst = os.path.join(SAVE_ROOT, subdir), os.path.join(DEST_ROOT, subdir)
    n_ok = n_bad = 0
    for p in glob.glob(src + '/**/*', recursive=True):
        if not os.path.isfile(p):
            continue
        d = os.path.join(dst, os.path.relpath(p, src))
        os.makedirs(os.path.dirname(d), exist_ok=True)
        try:
            if os.path.exists(d) and os.path.getsize(d) == os.path.getsize(p):
                n_ok += 1
                continue
            shutil.copy2(p, d)
            good = os.path.getsize(d) == os.path.getsize(p)
            n_ok += int(good)
            n_bad += int(not good)
        except Exception as e:
            n_bad += 1
            print('    gagal', os.path.relpath(p, src), e)
    print('  [{}] ke Drive: {} ok, {} gagal'.format(tag, n_ok, n_bad))
    return n_bad == 0

def cached(subdir, rel):
    """Sudah ada di Drive? Kalau ya, tarik ke lokal dan lewati GPU-nya."""
    d = os.path.join(DEST_ROOT, subdir, rel)
    if not (os.path.exists(d) and os.path.getsize(d) > 1000):
        return False
    l = os.path.join(SAVE_ROOT, subdir, rel)
    if not os.path.exists(l) or os.path.getsize(l) != os.path.getsize(d):
        os.makedirs(os.path.dirname(l), exist_ok=True)
        shutil.copy2(d, l)
    return True

print('pembungkus siap | tujuan Drive', DEST_ROOT)

## 7. Split bersih — direplikasi persis dari `unified_eval_corruption.py`

Repo membagi `random_split(clean, [5000, 20000, 25000], Generator().manual_seed(seed))`.
Permutasinya hanya bergantung pada panjang dan generator, jadi memanggilnya atas
`range(50000)` menghasilkan **indeks yang sama persis** — dan indeks itulah yang kita
butuhkan, karena logit baseline dihitung sekali untuk seluruh 50.000 lalu diiris per
seed. Tanpa itu, tiga seed berarti tiga kali kerja GPU untuk hasil yang identik.

In [ ]:
TTA_SIZE = int(0.2 * NUM_CAL)
CAL_SIZE = NUM_CAL - TTA_SIZE
LEFTOVER = len(clean_ds) - NUM_CAL

SPLITS = {}
for s in SEEDS:
    a, b, _ = tdata.random_split(range(len(clean_ds)),
                                 [TTA_SIZE, CAL_SIZE, LEFTOVER],
                                 generator=torch.Generator().manual_seed(s))
    SPLITS[s] = (np.array(a.indices), np.array(b.indices))
    assert len(set(a.indices) & set(b.indices)) == 0, 'D_TTA dan D_cal beririsan'
print('split per seed: D_TTA', TTA_SIZE, '| D_cal', CAL_SIZE, '| dibuang', LEFTOVER)
print('seed 0 vs 1 indeks D_cal identik?',
      bool((SPLITS[0][1] == SPLITS[1][1]).all()) if len(SEEDS) > 1 else 'n/a')

## 8. Logit baseline bersih — dihitung SEKALI untuk 50.000

`baseline` tidak memakai augmentasi dan tidak memakai parameter apa pun yang
bergantung seed, jadi logitnya untuk citra ke-i sama di ketiga seed. Menghitungnya
sekali dan mengindeksnya per seed bukan pengoptimalan yang berisiko: hasilnya bit-
identik dengan menghitung ulang, dan itu di-assert di bawah untuk satu irisan.

In [ ]:
CLEAN_BASE = SAVE_ROOT + '/_clean_baseline.pt'
if cached('', '_clean_baseline.pt'):
    print('logit baseline bersih diambil dari Drive')
else:
    t0 = time.time()
    lg, lb = extract_logits(clean_ds, 'baseline', None)
    save_pt(CLEAN_BASE, lg, lb)
    print('baseline bersih {:.0f}s | {}'.format(time.time() - t0, tuple(lg.shape)))
    push('', 'baseline bersih')

_cb = load_pt(CLEAN_BASE)
CLEAN_LOGITS, CLEAN_LABELS = _cb['logits'], _cb['labels'].long()
assert CLEAN_LOGITS.shape == (50000, 1000), CLEAN_LOGITS.shape
acc = float((CLEAN_LOGITS.argmax(1) == CLEAN_LABELS).float().mean())
print('akurasi top-1 val bersih: {:.4f}  (ResNet-50 torchvision ~0.761)'.format(acc))
assert 0.72 < acc < 0.80, 'akurasi di luar rentang wajar -- label mungkin tergeser'

## 9. Loop utama

Untuk tiap korupsi × severity: unduh → ekstrak → **verifikasi urutan kelas** → logit
baseline atas 50.000 citra terkorupsi → per seed simpan `baseline_cal.pt` (bersih) dan
`baseline_test.pt` (terkorupsi) + suhu Platt → UM-TTA bila diminta → salin ke Drive →
hapus citranya.

**Pemeriksaan urutan kelas itu yang paling penting di sel ini.** `ImageFolder`
mengurutkan kelas berdasarkan nama folder; kalau direktori terkorupsi punya himpunan
WNID yang sedikit berbeda, indeks label bergeser dan setiap angka setelahnya salah
tanpa satu pun error. Jadi daftarnya dibandingkan elemen per elemen, dan tidak cocok
= berhenti.

In [ ]:
import gc
MANIFESTS, SKIPPED = {}, []

def temp_of(logits, labels):
    return float(extract_temperature(model, tdata.TensorDataset(logits, labels.long()),
                                     alpha=0.05, batch_size=BATCH_SIZE))

def wanted_files(sev):
    """Berkas yang HARUS ada per seed untuk satu severity."""
    f = ['manifest.json', 'baseline_cal.pt']
    if DO_UMTTA and sev in UMTTA_SEVERITIES:
        f += ['umtta_cal.pt', 'umtta_test.pt']
    return f

def complete(corr, sev):
    """Sudah lengkap di Drive? Kalau ya, tar-nya tidak perlu diunduh sama sekali.

    Tanpa pemeriksaan ini, runtime yang mati lalu dijalankan ulang akan mengunduh
    24 GB lebih dulu, baru menyadari tidak ada yang perlu dikerjakan."""
    d = os.path.join(DEST_ROOT, '{}_s{}'.format(corr, sev))
    if not os.path.exists(os.path.join(d, '_corrupt_baseline.pt')):
        return False
    for seed in SEEDS:
        for f in wanted_files(sev):
            p = os.path.join(d, 'seed_{}'.format(seed), f)
            if not (os.path.exists(p) and os.path.getsize(p) > 100):
                return False
    return True

for corr, cat in CORRUPTIONS:
    todo = [sev for sev in SEVERITIES if not complete(corr, sev)]
    if not todo:
        print(corr + ': semua severity lengkap di Drive -- unduhan dilewati')
        continue
    if not fetch(corr, cat, todo):
        SKIPPED.append((corr, 'unduh/ekstrak gagal'))
        print('LEWATI', corr, '-- ekstraksi tidak lengkap')
        continue
    for sev in todo:
        sub = '{}_s{}'.format(corr, sev)
        print('\n' + '=' * 68)
        print(sub)
        print('=' * 68)
        cds = torchvision.datasets.ImageFolder(sev_dir(corr, sev), transform)
        assert list(cds.classes) == CLEAN_CLASSES, (
            'urutan kelas berbeda dari val bersih -- indeks label akan bergeser '
            'diam-diam. bersih {} vs korup {}'.format(
                CLEAN_CLASSES[:3], list(cds.classes)[:3]))
        print('  {} citra, urutan kelas cocok'.format(len(cds)))

        # --- baseline atas seluruh set terkorupsi: dihitung SEKALI, disimpan SEKALI.
        # Logit ini tidak bergantung seed, jadi menuliskannya per seed berarti tiga
        # salinan identik ~200 MB di Drive untuk setiap severity. Manifest tiap seed
        # menunjuk ke berkas yang satu ini lewat jalur relatif.
        rel_all = '_corrupt_baseline.pt'
        pth = os.path.join(SAVE_ROOT, sub, rel_all)
        if cached(sub, rel_all):
            print('  baseline terkorupsi dari Drive')
        else:
            t0 = time.time()
            lg, lb = extract_logits(cds, 'baseline', None)
            save_pt(pth, lg, lb)
            print('  baseline terkorupsi {:.0f}s'.format(time.time() - t0))
            del lg, lb; gc.collect()
        _c = load_pt(pth)
        C_LOGITS, C_LABELS = _c['logits'], _c['labels'].long()
        cacc = float((C_LOGITS.argmax(1) == C_LABELS).float().mean())
        print('  akurasi top-1 terkorupsi: {:.4f} (bersih {:.4f})'.format(cacc, acc))
        assert cacc < acc, 'korupsi tidak menurunkan akurasi -- citranya bersih?'

        for seed in SEEDS:
            sd = '{}/seed_{}'.format(sub, seed)
            _, cal_idx = SPLITS[seed]
            if all(cached(sub, 'seed_{}/{}'.format(seed, f))
                   for f in ('manifest.json', 'baseline_cal.pt')):
                MANIFESTS[sd] = json.load(open(
                    '{}/{}/manifest.json'.format(SAVE_ROOT, sd)))
                print('  seed {} baseline sudah ada'.format(seed))
                continue
            cl, cy = CLEAN_LOGITS[cal_idx], CLEAN_LABELS[cal_idx]
            save_pt('{}/{}/baseline_cal.pt'.format(SAVE_ROOT, sd), cl, cy)
            man = {'seed': seed, 'corruption': corr, 'severity': sev,
                   'cal_size': int(len(cal_idx)), 'test_size': int(len(C_LABELS)),
                   'cal_domain': 'clean', 'test_domain': 'corrupt',
                   'methods': ['baseline'],
                   'temperatures': {'baseline': temp_of(cl, cy)},
                   'files': {'baseline_cal': 'baseline_cal.pt',
                             'baseline_test': '../' + rel_all}}
            json.dump(man, open('{}/{}/manifest.json'.format(SAVE_ROOT, sd), 'w'),
                      indent=1)
            MANIFESTS[sd] = man
            print('  seed {} baseline siap | T={:.4f}'.format(
                seed, man['temperatures']['baseline']))
        push(sub, sub + ' baseline')
        del C_LOGITS, C_LABELS, _c; gc.collect()

        # --- UM-TTA: jembatan ke paper lama. Mahal, jadi hanya severity terpilih.
        if DO_UMTTA and sev in UMTTA_SEVERITIES:
            ty = np.array(cds.targets)
            per = max(1, UMTTA_TEST_N // len(CLEAN_CLASSES))
            rr = np.random.default_rng(42)   # tetap 42 di semua seed, supaya
            pick = np.sort(np.concatenate([  # perbandingan berpasangan atas baris identik
                rr.choice(np.where(ty == k)[0], min(per, int((ty == k).sum())),
                          replace=False)
                for k in range(len(CLEAN_CLASSES))]))
            test_sub = tdata.Subset(cds, pick.tolist())
            print('  umtta: subset test {} citra ({}/kelas)'.format(len(pick), per))
            for seed in SEEDS:
                sd = '{}/seed_{}'.format(sub, seed)
                if all(cached(sub, 'seed_{}/{}'.format(seed, f))
                       for f in ('umtta_cal.pt', 'umtta_test.pt')):
                    print('  seed {} umtta sudah ada'.format(seed))
                    continue
                tta_idx, cal_idx = SPLITS[seed]
                D_TTA = tdata.Subset(clean_ds, tta_idx.tolist())
                D_cal = tdata.Subset(clean_ds, cal_idx.tolist())
                t0 = time.time()
                torch.manual_seed(seed); np.random.seed(seed)
                torch.cuda.manual_seed(seed)
                theta = learn_global_tta_weights(model, D_TTA, NUM_AUG,
                                                 num_epochs=TTA_EPOCHS, lr=TTA_LR,
                                                 batch_size=BATCH_SIZE)
                w, b = learn_umtta_parameters(model, D_TTA, NUM_AUG, theta,
                                              num_epochs=TTA_EPOCHS, lr=TTA_LR,
                                              batch_size=BATCH_SIZE)
                par = {'theta_global': theta, 'w': w, 'b': b}
                ul, uy = extract_logits(D_cal, 'umtta', par)
                save_pt('{}/{}/umtta_cal.pt'.format(SAVE_ROOT, sd), ul, uy)
                tl, tyy = extract_logits(test_sub, 'umtta', par)
                save_pt('{}/{}/umtta_test.pt'.format(SAVE_ROOT, sd), tl, tyy)
                man = MANIFESTS.get(sd, {})
                if 'umtta' not in man.setdefault('methods', []):
                    man['methods'].append('umtta')
                man.setdefault('temperatures', {})['umtta'] = temp_of(ul, uy)
                man.setdefault('files', {}).update(
                    {'umtta_cal': 'umtta_cal.pt', 'umtta_test': 'umtta_test.pt'})
                man['umtta_test_size'] = int(len(tyy))
                man['umtta_test_indices_seed'] = 42
                json.dump(man, open('{}/{}/manifest.json'.format(SAVE_ROOT, sd), 'w'),
                          indent=1)
                json.dump({'w': w, 'b': b}, open(
                    '{}/{}/umtta_params.json'.format(SAVE_ROOT, sd), 'w'))
                torch.save(theta, '{}/{}/theta_global.pt'.format(SAVE_ROOT, sd))
                MANIFESTS[sd] = man
                print('  seed {} umtta {:.0f}s | T={:.4f}'.format(
                    seed, time.time() - t0, man['temperatures']['umtta']))
                del ul, uy, tl, tyy; gc.collect()
            push(sub, sub + ' umtta')
        del cds; gc.collect()
    # citra korupsi ini tidak dibutuhkan lagi; logitnya sudah ada di Drive
    shutil.rmtree('{}/{}'.format(IC_ROOT, corr), ignore_errors=True)
    print('citra', corr, 'dihapus')

## 10. Verifikasi apa yang benar-benar ada di Drive

Bukan mencetak niat, melainkan membaca kembali berkas yang tersimpan: bentuk tensor,
kecocokan jumlah baris dengan manifest, dan suhu yang tercatat. Notebook 12 akan
gagal dengan cara yang membingungkan kalau satu saja di antaranya meleset.

In [ ]:
rows = []
for man_p in sorted(glob.glob(DEST_ROOT + '/*/seed_*/manifest.json')):
    sd = os.path.dirname(man_p)
    man = json.load(open(man_p))
    # jalurnya dibaca DARI manifest, bukan ditebak dari pola nama: baseline_test
    # menunjuk ke ../_corrupt_baseline.pt yang dipakai bersama antar seed.
    for key, rel in sorted(man.get('files', {}).items()):
        p = os.path.normpath(os.path.join(sd, rel))
        if not os.path.exists(p):
            rows.append((os.path.relpath(man_p, DEST_ROOT) + ':' + key,
                         'HILANG', rel))
            continue
        d = load_pt(p)
        n, K = tuple(d['logits'].shape)
        exp = man['cal_size'] if key.endswith('_cal') else man['test_size']
        if key == 'umtta_test':
            exp = man.get('umtta_test_size')
        ok = (K == 1000 and n == len(d['labels']) and (exp is None or n == exp))
        rows.append((os.path.relpath(p, DEST_ROOT), '{}x{}'.format(n, K),
                     'OK' if ok else 'JUMLAH SALAH (harusnya {})'.format(exp)))
        del d
bad = [r for r in rows if r[2] != 'OK']
for r in rows[:12]:
    print('  {:52s} {:>12s} {}'.format(*r))
print('  ... total', len(rows), 'entri')
tot = sum(os.path.getsize(f) for f in glob.glob(DEST_ROOT + '/**/*', recursive=True)
          if os.path.isfile(f))
print('\nDrive: {:.2f} GB'.format(tot / 1e9))
print('dilewati:', SKIPPED if SKIPPED else 'tidak ada')
assert not bad, 'berkas bermasalah: ' + repr(bad[:5])
print()
print('CAVEAT: lengan baseline memakai 50.000 baris test (50/kelas) -> regime A.')
print('CAVEAT: lengan umtta memakai {} baris (10/kelas); statistik per-kelas TIDAK'
      .format(UMTTA_TEST_N))
print('        terukur di sana, jadi dibaca lewat bin prevalensi (regime B).')
print('CAVEAT: baseline_test dipakai bersama antar seed -- logitnya memang tidak')
print('        bergantung seed; yang berbeda hanya irisan kalibrasi bersihnya.')
print('CAVEAT: kalibrasi BERSIH, evaluasi TERKORUPSI -> exchangeability tidak berlaku.')
print('        Jaminan konformal batal untuk SEMUA metode di sini, bukan hanya PCC,')
print('        dan itu harus dijawab dengan literatur (weighted CP Tibshirani,')
print('        adaptive CP Gibbs-Candes) -- lihat audit F4.')
print()
print('SELESAI. Notebook 12 membaca', DEST_ROOT, 'tanpa GPU.')